<h1 style="display: flex; align-items: center; justify-content: center;">
  <img src="../../pictures/ai_circuits_dribbble.gif" width="100" style="margin-right: 10px;"/>
  🧹 Xử lý dữ liệu (Transformer Preprocessing)
</h1>


In [71]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn
import pickle
import os
import joblib


In [72]:
df = pd.read_csv(r"C:\Python\Data_Science_Atifficial_Interligent\project_datascience _AI\Data_Analsys\Building_a_Hybrid_Machine_Learning_Model_for_Bitcoin_Volatility_Prediction_in_the_Global_Crypto_Market\data\processed\Bitcoin_hybrid_features.csv")
df.drop(columns='Unnamed: 0',inplace=True)
df

,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI,BTC Dominance,Total CMC,USDTD,Bitcoin CMC,volatility_14_annual,garch_volatility
0,2018-02-01 00:00:00,10285.10,10335.00,10215.07,10263.18,594.441290,276.384030,0.414383,-157.324165,-60.240144,11710.140938,11732.920935,2018-02-01,30.0,38.130953,3.972854e+11,0.541264,1.514887e+11,1.561406,0.714646
1,2018-02-01 01:00:00,10263.18,10328.98,10216.00,10247.49,591.515725,275.414274,0.414405,-157.144845,-61.057994,11703.588708,11726.744507,2018-02-01,30.0,38.130953,3.972854e+11,0.541264,1.514887e+11,1.557828,0.714646
2,2018-02-01 02:00:00,10249.43,10317.73,10176.89,10199.61,479.412562,274.615613,0.414454,-156.598525,-62.014479,11696.921833,11720.394675,2018-02-01,30.0,38.130953,3.972854e+11,0.541264,1.514887e+11,1.556120,0.714646
3,2018-02-01 03:00:00,10199.61,10250.79,9959.04,10069.80,739.435309,274.717301,0.414138,-155.131977,-63.408817,11689.046417,11713.531496,2018-02-01,30.0,38.130953,3.972854e+11,0.541264,1.514887e+11,1.534316,0.714646
4,2018-02-01 04:00:00,10069.77,10256.00,10000.01,10245.79,649.939854,274.606160,0.413726,-158.422871,-64.105200,11681.703208,11707.428621,2018-02-01,30.0,38.130953,3.972854e+11,0.541264,1.514887e+11,1.534426,0.714646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70822,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0,59.352780,2.357494e+12,7.801736,1.399238e+12,0.613922,0.594760
70823,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12,0.613833,0.544528
70824,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12,0.599838,0.482215
70825,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12,0.600167,0.425413


In [73]:
def make_3d_data(data, window):
    X, Y = [], []
    for i in range(len(data)-window):
        # Lấy từ dòng i đến i+window, nhưng BỎ QUA cột cuối cùng ([:-1])
        # Điều này ép mô hình chỉ được dùng 7 cột (RSI, MACD, Volume...) làm Input
        X.append(data[i:i+window, :-1]) 
        
        # Target Y vẫn là cột số 7 (Volatility) của giờ tương lai
        Y.append(data[i+window][8])
        
    X = np.array(X)
    Y = np.array(Y)
    return X, Y

In [74]:
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
df = df.dropna()
df['timestamp'][169]

'2018-02-09 10:00:00'

In [75]:
new_df = df[['garch_volatility','log_return','ATR','BB_width_norm','volume','Total CMC','RSI','MACD_Hist','volatility_14_annual']]
new_df

,garch_volatility,log_return,ATR,BB_width_norm,volume,Total CMC,RSI,MACD_Hist,volatility_14_annual
1,0.714646,-0.001530,275.414274,0.414405,591.515725,3.972854e+11,-157.144845,-61.057994,1.557828
2,0.714646,-0.004683,274.615613,0.414454,479.412562,3.972854e+11,-156.598525,-62.014479,1.556120
3,0.714646,-0.012809,274.717301,0.414138,739.435309,3.972854e+11,-155.131977,-63.408817,1.534316
4,0.714646,0.017326,274.606160,0.413726,649.939854,3.972854e+11,-158.422871,-64.105200,1.534426
5,0.714646,-0.000761,273.463215,0.413547,387.674066,3.972854e+11,-158.332202,-64.794388,1.534336
...,...,...,...,...,...,...,...,...,...
70822,0.594760,0.003251,630.619773,0.125565,1165.159140,2.357494e+12,22715.462812,798.079696,0.613922
70823,0.544528,0.001375,629.331941,0.125669,728.929610,2.350619e+12,15875.054066,799.139303,0.613833
70824,0.482215,0.000377,627.163799,0.125801,947.248790,2.350619e+12,14663.557223,800.221240,0.599838
70825,0.425413,-0.003725,626.442292,0.125898,1612.875480,2.350619e+12,61217.518220,800.267129,0.600167


In [76]:
train_size = int(len(new_df)*0.8)
validate_size = int(len(new_df)*0.95)
train_set = new_df[:train_size]
validate_set = new_df[train_size:validate_size]
test_set = new_df[validate_size:]
(validate_set[0:168])
# test_set[168:].to_csv("test_set.csv")

,garch_volatility,log_return,ATR,BB_width_norm,volume,Total CMC,RSI,MACD_Hist,volatility_14_annual
56661,0.822061,-0.003492,474.479840,0.257678,676.67595,2.365789e+12,849.821979,645.102600,0.459720
56662,0.752698,-0.010599,476.829929,0.257109,1142.99300,2.365789e+12,1005.717808,642.964249,0.462506
56663,0.823428,0.000905,478.883224,0.256341,1184.21920,2.372684e+12,991.769921,641.000366,0.460293
56664,0.739456,0.001941,478.233650,0.255544,483.06066,2.372684e+12,963.152581,639.467600,0.460322
56665,0.670992,0.000873,476.484192,0.254859,260.89082,2.372684e+12,950.822054,638.090210,0.459206
...,...,...,...,...,...,...,...,...,...
56824,0.792495,-0.005532,517.094558,0.120395,1471.63779,2.106497e+12,-961.587703,-570.915370,0.501946
56825,0.750018,-0.005496,517.975065,0.120314,1887.75774,2.106497e+12,-886.266084,-585.533935,0.502598
56826,0.715310,0.000034,519.104233,0.120193,2092.03786,2.106497e+12,-886.740050,-599.821902,0.502320
56827,0.639510,0.002486,519.338867,0.120082,965.52159,2.106497e+12,-923.173882,-613.238532,0.502418


In [77]:
# train_set[168:].to_csv("train_set.csv")


In [78]:
# validate_set[168:].to_csv("validate_set.csv")


In [79]:
train_set_timestamp = df['timestamp'][:train_size]
train_set_timestamp = train_set_timestamp[168:]
validate_set_timestamp = df['timestamp'][train_size:validate_size]
validate_set_timestamp = validate_set_timestamp[168:]
test_set_timestamp = df['timestamp'][validate_size:]
test_set_timestamp = test_set_timestamp[168:]
len(train_set_timestamp)

56492

In [ ]:
scaler = StandardScaler()
scaler.fit(train_set)
train_set_scaled = scaler.transform(train_set)
test_set_scaled = scaler.transform(test_set)
validate_set_scaled = scaler.transform(validate_set)


SCALER_SAVE_PATH = r"C:\Python\Data_Science_Atifficial_Interligent\project_datascience _AI\Data_Analsys\Building_a_Hybrid_Machine_Learning_Model_for_Bitcoin_Volatility_Prediction_in_the_Global_Crypto_Market\src\model_save\scaler.pkl"

joblib.dump(scaler, SCALER_SAVE_PATH)
print(f"✅ Đã lưu Scaler thành công tại: {SCALER_SAVE_PATH}")

✅ Đã lưu Scaler thành công tại: C:\Python\Data_Science_Atifficial_Interligent\project_datascience _AI\Data_Analsys\Building_a_Hybrid_Machine_Learning_Model_for_Bitcoin_Volatility_Prediction_in_the_Global_Crypto_Market\src\model_save\scaler.pkl


In [84]:
train_set_scaled

array([[ 2.41061225e-01, -2.09875301e-01,  1.51269989e-01, ...,
         6.91050028e-04, -1.65611287e-01,  3.05037308e+00],
       [ 2.41061225e-01, -6.33158274e-01,  1.47607130e-01, ...,
         6.92114731e-04, -1.68097225e-01,  3.04475369e+00],
       [ 2.41061225e-01, -1.72383385e+00,  1.48073496e-01, ...,
         6.94972836e-04, -1.71721155e-01,  2.97301739e+00],
       ...,
       [-5.69023710e-01, -5.86935150e-01,  1.00054374e+00, ...,
         2.74888039e-03,  1.66964161e+00, -5.75044221e-01],
       [-5.36471151e-01, -9.02000440e-01,  1.03699991e+00, ...,
         2.95425497e-03,  1.66380364e+00, -5.70337209e-01],
       [-3.64853593e-01,  2.09524474e+00,  1.06744062e+00, ...,
         2.57271287e-03,  1.66807568e+00, -5.48903004e-01]])

In [85]:
X_train,Y_train = make_3d_data(train_set_scaled,168)
X_validate,Y_validate = make_3d_data(validate_set_scaled,168)
X_test, Y_test = make_3d_data(test_set_scaled,168)

In [86]:
X_train.shape

(56492, 168, 8)

In [87]:
train_set_timestamp

169      2018-02-09 10:00:00
170      2018-02-09 11:00:00
171      2018-02-09 12:00:00
172      2018-02-09 13:00:00
173      2018-02-09 14:00:00
                ...         
56656    2024-07-27 17:00:00
56657    2024-07-27 18:00:00
56658    2024-07-27 19:00:00
56659    2024-07-27 20:00:00
56660    2024-07-27 21:00:00
Name: timestamp, Length: 56492, dtype: object

In [89]:
train_set_timestamp

169      2018-02-09 10:00:00
170      2018-02-09 11:00:00
171      2018-02-09 12:00:00
172      2018-02-09 13:00:00
173      2018-02-09 14:00:00
                ...         
56656    2024-07-27 17:00:00
56657    2024-07-27 18:00:00
56658    2024-07-27 19:00:00
56659    2024-07-27 20:00:00
56660    2024-07-27 21:00:00
Name: timestamp, Length: 56492, dtype: object

In [90]:
# train_set_timestamp.to_csv("train_set_transformer_timestamp.csv")

In [91]:
Y_train.shape

(56492,)

In [92]:
X_validate.shape

(10456, 168, 8)

In [93]:
len(validate_set_timestamp)

10456

In [94]:
# validate_set_timestamp.to_csv("validate_set_transformer_timestamp.csv")


In [95]:
Y_validate.shape

(10456,)

In [96]:
X_test.shape

(3374, 168, 8)

In [97]:
# test_set_timestamp.to_csv("test_set_transformer_timestamp.csv")


In [99]:
Y_test.shape

(3374,)

In [100]:
with open("X_train.pkl","wb") as f:
    pickle.dump(X_train, f)

In [101]:
with open("Y_train.pkl","wb") as f:
    pickle.dump(Y_train, f)

In [102]:
with open("X_validate.pkl","wb") as f:
    pickle.dump(X_validate, f)

In [103]:
with open("Y_validate.pkl","wb") as f:
    pickle.dump(Y_validate, f)

In [104]:
with open("X_test.pkl","wb") as f:
    pickle.dump(X_test, f)

In [105]:
with open("Y_test.pkl","wb") as f:
    pickle.dump(Y_test, f)